In [1]:
import os
from utils.spark_session import get_spark_session

from pyspark.sql.functions import (
    round, avg, col, count, min, max, avg, when
)

In [2]:
def describe_column(df, categoric_column, numeric_column):
    stats_df = (
        df
        .groupBy(categoric_column)
        .agg(
            min(numeric_column).alias(f"min_{numeric_column}"),
            avg(numeric_column).alias(f"mean_{numeric_column}"),
            max(numeric_column).alias(f"max_{numeric_column}")
        )
    )
    
    return stats_df

In [3]:
spark = get_spark_session(app_name="01-eda-interim")

In [4]:
interim_df = spark.read.parquet(os.path.join('..', 'data', 'interim', 'interim.parquet'))

In [5]:
interim_df.show(5, truncate=False)

+--------------------------------+---------------------+------+------------+---------------------+-----------+----------------+---+-----------------+------+-------------+-------+---------+--------+-----+
|account_id                      |time_since_test_start|amount|max_discount|completed_offer_types|real_amount|target_converted|age|credit_card_limit|gender|registered_on|reg_day|reg_month|reg_year|index|
+--------------------------------+---------------------+------+------------+---------------------+-----------+----------------+---+-----------------+------+-------------+-------+---------+--------+-----+
|7da25b87262f4c75bac253cf5e5d9039|10.5                 |32.02 |NULL        |NULL                 |32.02      |0               |69 |109000.0         |F     |2017-04-06   |6      |4        |2017    |0    |
|e88b9aaa938c446289fb909cffeaa255|13.25                |25.09 |NULL        |NULL                 |25.09      |0               |76 |114000.0         |F     |2015-08-06   |6      |8     

In [6]:
interim_df.filter(col("age") < 0).show(5, truncate=False)

+--------------------------------+---------------------+------+------------+---------------------+-----------+----------------+---+-----------------+-------+-------------+-------+---------+--------+-----+
|account_id                      |time_since_test_start|amount|max_discount|completed_offer_types|real_amount|target_converted|age|credit_card_limit|gender |registered_on|reg_day|reg_month|reg_year|index|
+--------------------------------+---------------------+------+------------+---------------------+-----------+----------------+---+-----------------+-------+-------------+-------+---------+--------+-----+
|ffecb1f8543f4bf7bade023de366d6bf|25.0                 |0.36  |NULL        |NULL                 |0.36       |0               |-1 |-1.0             |unknown|2017-10-27   |27     |10       |2017    |9    |
|2a06fdb4f40749cb8830973a99e9672a|11.0                 |2.43  |NULL        |NULL                 |2.43       |0               |-1 |-1.0             |unknown|2015-10-21   |21     |1

In [7]:
interim_df.groupBy("target_converted").count().show()

+----------------+------+
|target_converted| count|
+----------------+------+
|               1| 30617|
|               0|108336|
+----------------+------+



Classes desbalanceadas, as ofertas convertidas representam apenas 22% das transações.

In [8]:
(
    interim_df.withColumn(
        "real_amount_group",
        when(col("real_amount") < 0, "Unknown")
        .when(col("real_amount") < 50, "0-49")
        .when(col("real_amount") < 100, "50-99")
        .when(col("real_amount") < 150, "100-149")
        .when(col("real_amount") < 200, "150-199")
        .when(col("real_amount") < 250, "200-249")
        .when(col("real_amount") < 300, "250-299")
        .otherwise("300+")
    )
    .groupBy("real_amount_group")
    .agg(round(avg("target_converted"), 3).alias("conversion_rate"))
    .orderBy("real_amount_group")
).orderBy("conversion_rate").show(truncate=False)


+-----------------+---------------+
|real_amount_group|conversion_rate|
+-----------------+---------------+
|0-49             |0.218          |
|150-199          |0.364          |
|250-299          |0.375          |
|300+             |0.399          |
|100-149          |0.604          |
|200-249          |0.667          |
|50-99            |0.742          |
+-----------------+---------------+



A faixa com valor total da transação possui uma taxa de conversão superior. Mesmo com a oferta, o cliente ainda gasta mais. 

In [9]:
interim_df.filter(col("age") < 0).groupBy("completed_offer_types").count().orderBy("count").show()

+---------------------+-----+
|completed_offer_types|count|
+---------------------+-----+
|        discount,bogo|   39|
|                 bogo|  346|
|             discount|  651|
|                 NULL|13960|
+---------------------+-----+



*Discount* é o tipo de cupom mais utilizado.

In [10]:
(
    interim_df.groupBy("gender", "target_converted")
    .agg(count("*").alias("n"))
    .groupBy("gender")
    .pivot("target_converted")
    .sum("n")
    .withColumnRenamed("0", "not_converted")
    .withColumnRenamed("1", "converted")
    .fillna(0)
    .withColumn("conversion_rate", round(col("converted") / (col("converted") + col("not_converted")), 3))
).show()


+-------+-------------+---------+---------------+
| gender|not_converted|converted|conversion_rate|
+-------+-------------+---------+---------------+
|      F|        35308|    14074|          0.285|
|unknown|        13960|     1036|          0.069|
|      M|        57747|    15047|          0.207|
|      O|         1321|      460|          0.258|
+-------+-------------+---------+---------------+



As proporções de sexo entre cliente convertidos ou não são bem balanceadas.

In [11]:
(
    interim_df.withColumn(
        "age_group",
        when(col("age") < 0, "Unknown")
        .when(col("age") < 10, "0-9")
        .when(col("age") < 20, "10–19")
        .when(col("age") < 30, "20–29")
        .when(col("age") < 40, "30–39")
        .when(col("age") < 50, "40–49")
        .when(col("age") < 60, "50–59")
        .when(col("age") < 70, "60–69")
        .when(col("age") < 80, "70–79")
        .otherwise("80+")
    )
    .groupBy("age_group")
    .agg(round(avg("target_converted"), 3).alias("conversion_rate"))
    .orderBy("age_group")
).show(truncate=False)

+---------+---------------+
|age_group|conversion_rate|
+---------+---------------+
|10–19    |0.142          |
|20–29    |0.155          |
|30–39    |0.181          |
|40–49    |0.233          |
|50–59    |0.272          |
|60–69    |0.267          |
|70–79    |0.273          |
|80+      |0.275          |
|Unknown  |0.069          |
+---------+---------------+



As faixas acima de 40 anos concentram uma proporção interessante de clientes convertidos. 

In [12]:
(
    interim_df.groupBy("reg_year")
    .agg(round(avg("target_converted"), 3).alias("conversion_rate"))
    .orderBy("reg_year")
).show()


+--------+---------------+
|reg_year|conversion_rate|
+--------+---------------+
|    2013|          0.147|
|    2014|          0.147|
|    2015|           0.21|
|    2016|          0.241|
|    2017|          0.229|
|    2018|          0.216|
+--------+---------------+



In [13]:
(
    interim_df.groupBy("reg_month")
    .agg(round(avg("target_converted"), 3).alias("conversion_rate"))
    .orderBy("reg_month")
).show()


+---------+---------------+
|reg_month|conversion_rate|
+---------+---------------+
|        1|          0.216|
|        2|          0.217|
|        3|          0.215|
|        4|          0.221|
|        5|          0.222|
|        6|          0.218|
|        7|          0.215|
|        8|          0.221|
|        9|          0.226|
|       10|          0.224|
|       11|          0.223|
|       12|          0.225|
+---------+---------------+



Mês e ano não possuem impacto na conversão, não é possível levantar hipótese de sazonalidade.

In [14]:
(
    interim_df.groupBy("target_converted")
    .agg(round(avg("real_amount"), 2).alias("avg_real_amount"))
).show()


+----------------+---------------+
|target_converted|avg_real_amount|
+----------------+---------------+
|               1|          25.17|
|               0|           10.7|
+----------------+---------------+



A taxa de conversão é alta para pessoas que gastam mais no aplicativo.

In [15]:
(
    interim_df.groupBy("target_converted")
    .agg(round(avg("time_since_test_start"), 2).alias("time_since_test_start"))
).show()


+----------------+---------------------+
|target_converted|time_since_test_start|
+----------------+---------------------+
|               1|                16.28|
|               0|                15.79|
+----------------+---------------------+



O tempo da promoção possui leve impacto na conversão.

In [16]:
spark.stop()